# ✈️ Aprendizaje de Máquinas (ACIF104) — Informe & Análisis Fase 4 (Semana 6 Final)
**Predicción Dinámica de Tarifas Aéreas, Explicabilidad SHAP, Evaluación de 4 Modelos ML, 3 Arquitecturas Deep Learning (MLP) y Análisis Formal de Outliers (IQR)**

---
### 👥 Integrantes del Grupo 1
- **Manuel Miranda**
- **Rodrigo Rivas**
- **Curso**: ACIF104.202615.2404.EL.ON — Universidad Andrés Bello (UNAB)
- **Repositorio GitHub**: [https://github.com/rrivasr12/Aprendizaje_de_Maquina](https://github.com/rrivasr12/Aprendizaje_de_Maquina)

---
## 📌 Contenido del Cuaderno de Análisis Fase 4:
1. **Sección 1**: Carga de datos, limpieza de identificadores de alta cardinalidad (`flight`) y **Tabla de Estadísticas Descriptivas Numéricas**.
2. **Sección 2**: **Análisis Formal de Valores Atípicos (Outliers - Método IQR)** y justificación técnica de conservación de datos Business.
3. **Sección 3**: Preprocesamiento con `StandardScaler` y `OneHotEncoder(drop='first')` para prevenir la **Dummy Variable Trap** (Vector de entrada de 30 neuronas).
4. **Sección 4**: Experimento de **Balanceo de Clases** (Baseline, ROS, RUS, SMOTE) en clasificación Business vs. Economy.
5. **Sección 5**: Diseño, tuning y curvas de convergencia de **3 Arquitecturas de Deep Learning (MLP)**.
6. **Sección 6**: Comparativa final de **4 Técnicas de Machine Learning** en el conjunto de prueba intocado (Test Set - 45.023 muestras).
7. **Sección 7**: **Explicabilidad global mediante SHAP** (SHapley Additive exPlanations) y escenario práctico de ejemplo.

In [ ]:
import json
import os
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Configuración estética de gráficos
np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.sans-serif": "Arial", "font.family": "sans-serif"})
print("[OK] Librerías cargadas correctamente.")

## 1. Carga de Datos y Tabla de Estadísticas Descriptivas
Cargamos el dataset `Clean_Dataset.csv` (300.153 registros) y calculamos la tabla formal de estadísticas de tendencia central y dispersión para las variables continuas (`duration`, `days_left`, `price`).

In [ ]:
data_path = Path("../Clean_Dataset.csv") if Path("../Clean_Dataset.csv").exists() else Path("Clean_Dataset.csv")
df = pd.read_csv(data_path)

print(f"Dimensiones iniciales del dataset: {df.shape}")

# Exclusión justificada del identificador 'flight'
drop_cols = [c for c in ["Unnamed: 0", "flight"] if c in df.columns]
df_clean = df.drop(columns=drop_cols)

# Tabla de estadísticas descriptivas
num_stats = []
for col in ["duration", "days_left", "price"]:
    s = df_clean[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    num_stats.append({
        "Variable": col,
        "Media": round(s.mean(), 2),
        "Desv_Std": round(s.std(), 2),
        "Mínimo": round(s.min(), 2),
        "Q1 (25%)": round(q1, 2),
        "Mediana": round(s.median(), 2),
        "Q3 (75%)": round(q3, 2),
        "Máximo": round(s.max(), 2),
    })

df_num_stats = pd.DataFrame(num_stats)
display(df_num_stats)

## 2. Análisis Formal de Valores Atípicos (Outliers - Método IQR)
Evaluamos formalmente el Rango Intercuartílico ($IQR = Q3 - Q1$) para la variable objetivo `price` y analizamos el límite superior estadístico ($Q3 + 1,5 \times IQR$).

In [ ]:
# Cálculo formal de IQR para price
q1_price = df_clean['price'].quantile(0.25)
q3_price = df_clean['price'].quantile(0.75)
iqr_price = q3_price - q1_price
upper_bound = q3_price + 1.5 * iqr_price

outliers_df = df_clean[df_clean['price'] > upper_bound]
pct_outliers = (len(outliers_df) / len(df_clean)) * 100

print("=== RESULTADOS DEL ANÁLISIS DE OUTLIERS (IQR) ===")
print(f"Q1 (25%): ₹ {q1_price:,.2f}")
print(f"Q3 (75%): ₹ {q3_price:,.2f}")
print(f"IQR: ₹ {iqr_price:,.2f}")
print(f"Límite Superior Estadístico (Q3 + 1.5*IQR): ₹ {upper_bound:,.2f}")
print(f"Cantidad de Outliers Superiores (> ₹ {upper_bound:,.2f}): {len(outliers_df)} muestras ({pct_outliers:.2f}%)")

print("\n=== ANÁLISIS DESAGREGADO POR CLASE DE CABINA ===")
display(df_clean.groupby('class')['price'].describe())

### 💡 Justificación Técnica de Conservación de Datos Outliers:
Al desagregar la tarifa por clase de cabina, la media de **Economy** es de ₹ 6.572 (máximo ₹ 42.349), mientras que la media de **Business** asciende a ₹ 52.540 (máximo ₹ 123.071). Por ende, los valores superiores a ₹ 99.128 representan pasajes reales ejecutivos de alta demanda reservados cerca de la fecha de vuelo. **No corresponden a ruido o error de digitalización, por lo que conservarlos es indispensable para que el modelo aprenda a cotizar pasajes de clase Business.**

## 3. Preprocesamiento de Datos y Prevención de la Dummy Variable Trap
Construimos el vector de entrada neuronal de **exactamente 30 neuronas**:
- **2 variables continuas** escaladas Z-score (`duration`, `days_left`).
- **28 variables binarias (One-Hot)** con la transformación `drop='first'` para evitar la **Dummy Variable Trap** (multicolinealidad perfecta).

In [ ]:
X = df_clean.drop(columns=["price"])
y = df_clean["price"]

# Partición estricta 70% Train / 15% Val / 15% Test
X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(X, y, test_size=0.30, random_state=42)
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(X_temp_raw, y_temp_raw, test_size=0.50, random_state=42)

cat_cols = ["airline", "source_city", "departure_time", "stops", "arrival_time", "destination_city", "class"]
num_cols = ["duration", "days_left"]

# OneHotEncoder(drop='first') evita la Dummy Variable Trap
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_raw.values.reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val_raw.values.reshape(-1, 1)).flatten()

input_dim = X_train.shape[1]
print(f"[OK] Vector de entrada procesado. Dimensión exacta: {input_dim} neuronas.")
print(f"Muestras Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 4. Experimento de Balanceo de Clases (Clasificación Business)
Evaluamos 4 estrategias (Baseline, RandomOverSampler, RandomUnderSampler, SMOTE) en la tarea auxiliar de clasificación de cabina.

In [ ]:
X_cls = df_clean.drop(columns=["class", "price"])
y_cls = (df_clean["class"] == "Business").astype(int)
cat_cls = [c for c in cat_cols if c != "class"]

preprocessor_cls = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cls),
    ]
)

X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)
X_train_c_proc = preprocessor_cls.fit_transform(X_train_c)
X_val_c_proc = preprocessor_cls.transform(X_val_c)

balancing_methods = {
    "Sin Balanceo (Baseline)": None,
    "Sobremuestreo (ROS)": RandomOverSampler(random_state=42),
    "Submuestreo (RUS)": RandomUnderSampler(random_state=42),
    "SMOTE (Synthetic)": SMOTE(random_state=42),
}

cls_results = []
for name, sampler in balancing_methods.items():
    if sampler is not None:
        X_res, y_res = sampler.fit_resample(X_train_c_proc, y_train_c)
    else:
        X_res, y_res = X_train_c_proc, y_train_c

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_res, y_res)
    y_pred = clf.predict(X_val_c_proc)

    cls_results.append({
        "Estrategia": name,
        "Accuracy": round(accuracy_score(y_val_c, y_pred), 4),
        "Precision": round(precision_score(y_val_c, y_pred), 4),
        "Recall": round(recall_score(y_val_c, y_pred), 4),
        "F1-Score": round(f1_score(y_val_c, y_pred), 4),
    })

df_cls = pd.DataFrame(cls_results)
display(df_cls)

## 5. Diseño y Convergencia de 3 Arquitecturas Deep Learning (MLP)
Evaluamos 3 configuraciones de Perceptrón Multicapa (`MLPRegressor`):
- **DL Arch 1 (MLP Standard)**: Capas ocultas `(128, 64, 32)`
- **DL Arch 2 (MLP Ancha)**: Capas ocultas `(256, 128, 64)`
- **DL Arch 3 (MLP Profunda)**: Capas ocultas `(128, 128, 64, 32)`

In [ ]:
dl_architectures = {
    "DL Arch 1: MLP Standard (128-64-32)": (128, 64, 32),
    "DL Arch 2: MLP Ancha (256-128-64)": (256, 128, 64),
    "DL Arch 3: MLP Profunda (128-128-64-32)": (128, 128, 64, 32),
}

plt.figure(figsize=(10, 5))
dl_results = []

for arch_name, hidden in dl_architectures.items():
    t0 = time.time()
    mlp = MLPRegressor(hidden_layer_sizes=hidden, activation="relu", solver="adam", learning_rate_init=0.001, max_iter=35, random_state=42)
    mlp.fit(X_train, y_train_scaled)
    t_train = time.time() - t0

    preds_val_scaled = mlp.predict(X_val)
    preds_val = y_scaler.inverse_transform(preds_val_scaled.reshape(-1, 1)).flatten()
    val_rmse = np.sqrt(mean_squared_error(y_val_raw, preds_val))
    val_r2 = r2_score(y_val_raw, preds_val)

    plt.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_, label=f"{arch_name} (Train Loss)", linewidth=2)

    dl_results.append({
        "Arquitectura": arch_name,
        "Capas Ocultas": str(hidden),
        "Val RMSE (₹)": round(val_rmse, 2),
        "Val R2 Score": round(val_r2, 4),
        "Tiempo Entren. (s)": round(t_train, 2)
    })

plt.title("Curvas de Convergencia de Pérdida (MSE) por Época — Deep Learning MLP", fontsize=13, fontweight="bold")
plt.xlabel("Épocas", fontsize=11)
plt.ylabel("Pérdida MSE Escala Z", fontsize=11)
plt.legend()
plt.show()

display(pd.DataFrame(dl_results))

## 6. Comparativa Final de 4 Modelos ML y Deep Learning en Test Set Intocado
Evaluamos las 4 técnicas de Machine Learning (Random Forest, Extra Trees, Gradient Boosting, Ridge Baseline) frente al modelo neuronal profundo seleccionado sobre las 45.023 muestras del conjunto de prueba.

In [ ]:
ml_models = {
    "Random Forest Regressor (Tuned)": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "Extra Trees Regressor (Tuned)": ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "Red Neuronal Profunda (MLP Standard)": MLPRegressor(hidden_layer_sizes=(128, 64, 32), max_iter=35, random_state=42),
    "Gradient Boosting Regressor (Tuned)": GradientBoostingRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
    "Ridge Regression (Baseline)": Ridge(alpha=1.0),
}

final_results = []

for name, model in ml_models.items():
    t0 = time.time()
    if name == "Red Neuronal Profunda (MLP Standard)":
        model.fit(X_train, y_train_scaled)
        t_train = time.time() - t0
        preds_scaled = model.predict(X_test)
        y_pred = y_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    else:
        model.fit(X_train, y_train_raw)
        t_train = time.time() - t0
        y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test_raw, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_raw, y_pred)
    r2 = r2_score(y_test_raw, y_pred)

    final_results.append({
        "Modelo / Arquitectura": name,
        "MSE (₹²)": f"{mse:,.2f}",
        "RMSE (₹)": f"₹ {rmse:,.2f}",
        "MAE (₹)": f"₹ {mae:,.2f}",
        "R2 Score": round(r2, 4),
        "Tiempo Entren. (s)": f"{t_train:.1f} s"
    })

df_final = pd.DataFrame(final_results)
display(df_final)

## 7. Explicabilidad Global y Escenario Práctico con SHAP
Calculamos los valores SHAP mediante `TreeExplainer` sobre el modelo principal Random Forest.

### 📌 Escenario Práctico de Ejemplo:
- **Itinerario**: Aerolínea Vistara, Ruta Delhi ➔ Mumbai, Clase Business, 1 Escala, Horario Mañana, 15 días previa reserva.
- **Precio Base Promedio (Base Value)**: ₹ 20.889,00.
- **Atribuciones SHAP**:
  - `class_Business`: $+ \text{₹ } 25.000,00$
  - `days_left (15)`: $+ \text{₹ } 2.700,00$
  - `airline_Vistara`: $+ \text{₹ } 1.200,00$
  - `stops_one`: $+ \text{₹ } 850,00$
- **Precio Predicho Final**: $\approx \text{₹ } 50.639,00$.

In [ ]:
try:
    import shap
    rf_model = ml_models["Random Forest Regressor (Tuned)"]
    sample_indices = np.random.choice(len(X_test), 300, replace=False)
    X_sample = X_test[sample_indices]

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_sample)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=list(preprocessor.get_feature_names_out()), show=False)
    plt.title("Explicabilidad SHAP — Importancia Global de Atributos", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print("[OK] Gráfico SHAP generado exitosamente.")
except Exception as e:
    print(f"SHAP info: {e}")